# Lösung: Modell exportieren

Für den Einsatz auf anderen Rechnern, im Browser oder auf Edge-Geräten wird das Modell in ein anderes Format exportiert, z. B. **ONNX**.

## Umgebung

Die Notebooks laufen lokal oder in Google Colab. Für Colab `use_colab = True` setzen und unter *Laufzeit > Laufzeittyp ändern* eine GPU wählen.

Die Version von `ultralytics` ist festgelegt, damit die Notebooks auch in späteren Semestern laufen.

In [ ]:
use_colab = False

if use_colab:
    %pip install -q ultralytics==8.4.215
    from google.colab import drive
    drive.mount("/content/drive")
    base_path = "/content/drive/MyDrive"
else:
    # lokal einmalig installieren: pip install ultralytics==8.4.215
    base_path = ".."

import ultralytics
ultralytics.checks()

Für den ONNX-Export installiert `ultralytics` die Pakete `onnx` und `onnxslim` normalerweise selbst. Falls das scheitert: `pip install onnx onnxslim onnxruntime`.

In [ ]:
from ultralytics import YOLO

MODEL_PATH = "runs/detect/muenzen/weights/best.pt"
model = YOLO(MODEL_PATH)
onnx_file = model.export(format="onnx", imgsz=960)
print("Exportiert nach", onnx_file)

Das exportierte Modell wird genauso verwendet wie das PyTorch-Modell.

In [ ]:
import os
import time
from pathlib import Path

test_dir = Path(base_path) / "Datensätze" / "muenzen_gesamt" / "images" / "test"
images = [str(p) for p in sorted(test_dir.glob("*.jpg"))[:20]]

for name, m in [("PyTorch", YOLO(MODEL_PATH)), ("ONNX", YOLO(onnx_file, task="detect"))]:
    m.predict(images[0], verbose=False)             # Aufwärmen
    start = time.perf_counter()
    count = sum(len(m.predict(b, verbose=False)[0].boxes) for b in images)
    duration = (time.perf_counter() - start) / len(images) * 1000
    print(f"{name:8s}: {duration:6.1f} ms pro Bild, {count} Detektionen")

## Weitere Formate

| Ziel | Format |
|---|---|
| Plattformunabhängig | `onnx` |
| NVIDIA-GPU, Jetson | `engine` (TensorRT) |
| Intel-CPU | `openvino` |
| Android, Raspberry Pi | `tflite` |
| iPhone, Mac | `coreml` |

Mit `int8=True` werden die Gewichte auf 8 Bit quantisiert: kleiner und schneller, meist mit geringem Verlust an Genauigkeit. Mit `nms=False` wird bei YOLO26 der NMS-freie Head exportiert.

In [ ]:
# model.export(format="openvino", int8=True, data="../Datensätze/muenzen_gesamt/data.yaml")
# model.export(format="tflite")
# model.export(format="onnx", nms=False)